# పాఠం 16 - Microsoft Foundryతో స్కేలబుల్ ఏజెంట్లను అమలు చేయడం

ఈ నోటుబుక్‌లో మీరు కల్పనాత్మక సంస్థ **Contoso** కోసం **ప్రొడక్షన్-సిద్ధంగా ఉన్న కస్టమర్ సపోర్ట్ ఏజెంట్** ని నిర్మిస్తారు. ముందు పాఠాల నుండి వేరుగా, ఇక్కడ ముఖ్య విషయం ఏజెంట్ యొక్క తర్క చక్రం కాదు — అది ఒక ఏజెంట్‌ను పెద్ద ఎత్తున సురక్షితంగా నడపడానికి చుట్టూ ఉన్న **ప్రతీ అంశం**:

1. **టూల్ కాలింగ్** — ఆర్డర్ లుకప్స్ మరియు టికెట్ సృష్టి.
2. **RAG** — జ్ఞాన ఆధారము నుండి విధాన సమాధానాలు.
3. **మెమరీ** — టర్న్‌ల మధ్య కస్టమర్‌ను గుర్తు పెట్టుకోవడం.
4. **మోడల్ రూటింగ్** — సులభమైన అభ్యర్థనలను చిన్న మోడల్‌కి పంపడం, క్లిష్టమైన వాటిని పెద్ద మోడల్‌కి పంపడం.
5. **స్పందన క్యాషింగ్** — మోడల్ కాల్ లేకుండా తిరిగి వచ్చే ప్రశ్నలకు సేవ ఇవ్వడం.
6. **మానవ ఆమోదం** — నిర్దిష్ట పరిమితిని మించి రిఫండ్‌లు ఒప్పందానికి నిలిపివేయడం.
7. **ఇవాల్యుయేషన్ గేట్** — చెత్త విడుదలను అడ్డుకుంటూ ఆఫ్లైన్ పరీక్ష సెట్.
8. **నిరీక్షణ** — ప్రతి అభ్యర్థన చుట్టూ OpenTelemetry ట్రేసింగ్.

ప్రతి విభాగం స్వతంత్రంగా కొనసాగించగలదు. ప్రతి వాక్యాన్ని చదవండి — ప్రొడక్షన్ ప్రిమిటివ్స్ జాగ్రత్తగా చిన్నవిగా ఉంచబడ్డాయి.


## సెటప్

ఈ నోట్‌బుక్‌ను నడిపించాక ముందు, మీరు నిర్ధారించుకోండి:

1. **మైক্রోసాఫ్ట్ ఫౌన్డ్రీ ప్రాజెక్ట్** ఒక నియోజక చాట్ మోడల్ (ఉదా. `gpt-5-mini`)తో.
2. **ఆజ్యూర్ CLI తో లాగిన్ అయి ఉండాలి** — మీ టెర్మినల్లో `az login` నడపండి.
3. **అవసరమైన వాతావరణ변数లను సెట్ చేయాలి:**
   - `AZURE_AI_PROJECT_ENDPOINT` — మీ మైক্রోసాఫ్ట్ ఫౌన్డ్రీ ప్రాజెక్ట్ ఎండ్పాయింట్.
   - `AZURE_AI_MODEL_DEPLOYMENT_NAME` — మీ నియోజక మోడల్ పేరు.

RAG విభాగం **Azure AI Search** ఉపయోగిస్తుంది, ఎప్పుడు `AZURE_SEARCH_SERVICE_ENDPOINT` మరియు `AZURE_SEARCH_API_KEY` సెట్ అవుతాయి, లేదంటే ఇది ఇన్-మెమరీ సెర్చ్‌ను వాడి నోట్‌బుక్ సెర్చ్ వనరుల్లేకుండా నడుస్తుంది.


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import re
import dotenv
from typing import Annotated

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential

dotenv.load_dotenv(dotenv.find_dotenv())

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
model = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

if not endpoint or not model:
    raise ValueError(
        "Missing required environment variables. "
        "Please set AZURE_AI_PROJECT_ENDPOINT and AZURE_AI_MODEL_DEPLOYMENT_NAME in your .env file."
    )

provider = FoundryChatClient(
    project_endpoint=endpoint,
    model=model,
    credential=AzureCliCredential(),
)
print("Foundry client ready.")

## 1. సాధనాలు

ఉత్పత్తి సాధనాలు నిజమైన వ్యవస్థలపై నిజమైన పని చేస్తాయి. ఇక్కడ మేము సాధారణ Python ఫంక్షన్లతో ఒక ఆర్డర్ డేటాబేస్ మరియు ఒక టిక్కెటింగ్ సిస్టమ్‌ను అనుకరించాము. `@tool` డెకొరేటర్ వాటిని ఏజెంట్‌కు పరిచయం చేస్తుంది.

`issue_refund` ఓ రిఫండ్ల కోసం `approval_mode="always_require"` ఉపయోగిస్తుందని గమనించండి — ఇది మేము తర్వాత అమలు చేసే మానవ-ఇన్-ది-లూప్ ప్రిమిటివ్.


In [ ]:
# Simulated backend systems (in production these are API calls behind scoped identities).
ORDERS = {
    "A1001": {"status": "shipped", "total": 42.00, "eta": "2 days"},
    "A1002": {"status": "processing", "total": 128.50, "eta": "5 days"},
    "A1003": {"status": "delivered", "total": 19.99, "eta": "delivered"},
}
TICKETS: list[dict] = []
REFUND_APPROVAL_THRESHOLD = 50.0


@tool(approval_mode="never_require")
def get_order_status(order_id: Annotated[str, "The customer's order ID, e.g. A1001"]) -> str:
    """Look up the status of a customer order."""
    order = ORDERS.get(order_id.upper())
    if not order:
        return f"No order found with ID {order_id}."
    return (
        f"Order {order_id.upper()}: status={order['status']}, "
        f"total=${order['total']:.2f}, eta={order['eta']}."
    )


@tool(approval_mode="never_require")
def open_ticket(
    subject: Annotated[str, "Short subject line for the support ticket"],
    details: Annotated[str, "Full description of the customer's issue"],
) -> str:
    """Open a support ticket for issues that need human follow-up."""
    ticket_id = f"T{1000 + len(TICKETS) + 1}"
    TICKETS.append({"id": ticket_id, "subject": subject, "details": details})
    return f"Ticket {ticket_id} opened: {subject}"


def refund_needs_approval(amount: float) -> bool:
    """Refunds above the threshold require a human approver."""
    return amount > REFUND_APPROVAL_THRESHOLD


@tool(approval_mode="always_require")
def issue_refund(
    order_id: Annotated[str, "The order to refund"],
    amount: Annotated[float, "Refund amount in USD"],
) -> str:
    """Issue a refund. Execution pauses for human approval before it runs."""
    return f"Refund of ${amount:.2f} issued for order {order_id.upper()}."


print("Tools defined.")

## 2. RAG — విధాన జ్ఞానభాండారం  

విధాన ప్రశ్నలకు ("మీ రిటర్న్ విండో ఎంత?" వంటి) సమాధానాలు మోడల్ యొక్క జ్ఞాపకశక్తి నుండి కాకుండా, అధికారం అధారంగా ఉన్న మూలం నుండి ఇవ్వబడాలి. మేము ఒక చిన్న జ్ఞానభాండారాన్ని శోధన టూల్‌గా చుట్టిపిడి ఉంటాము.  

ఉత్పత్తిలో ఇది **Azure AI Search**; ఇక్కడ మేము ఒక స్మృతి లోపు కీవర్డ్ శోధనను అందిస్తాము కాబట్టి నోట్‌బుక్ ఎక్కడైనా నడుస్తుంది, మరియు వాతావరణ మార్పునలు ఉన్నప్పుడు స్వయంచాలకంగా Azure AI Search కి మారుతుంది.  


In [ ]:
KNOWLEDGE_BASE = {
    "returns": "Contoso accepts returns within 30 days of delivery for a full refund. Items must be unused and in original packaging.",
    "shipping": "Standard shipping takes 3-5 business days. Express shipping (1-2 days) is available at checkout for an extra fee.",
    "warranty": "All Contoso electronics carry a 12-month limited warranty covering manufacturing defects.",
    "refund_policy": "Refunds are processed to the original payment method within 5 business days of approval. Refunds over $50 require a supervisor's approval.",
}


def _in_memory_search(query: str) -> str:
    q = query.lower()
    hits = [text for key, text in KNOWLEDGE_BASE.items() if key.replace("_", " ") in q or key in q]
    if not hits:
        # crude keyword fallback so the tool still returns something useful
        hits = [text for text in KNOWLEDGE_BASE.values() if any(w in text.lower() for w in q.split())]
    return "\n".join(hits) if hits else "No matching policy found."


def _azure_search(query: str) -> str:
    from azure.core.credentials import AzureKeyCredential
    from azure.search.documents import SearchClient

    client = SearchClient(
        endpoint=os.environ["AZURE_SEARCH_SERVICE_ENDPOINT"],
        index_name=os.getenv("AZURE_SEARCH_INDEX_NAME", "contoso-policies"),
        credential=AzureKeyCredential(os.environ["AZURE_SEARCH_API_KEY"]),
    )
    results = client.search(search_text=query, top=3)
    return "\n".join(r.get("content", "") for r in results) or "No matching policy found."


USE_AZURE_SEARCH = bool(os.getenv("AZURE_SEARCH_SERVICE_ENDPOINT") and os.getenv("AZURE_SEARCH_API_KEY"))


@tool(approval_mode="never_require")
def search_policies(query: Annotated[str, "The policy question to look up"]) -> str:
    """Search Contoso support policies to answer customer questions."""
    if USE_AZURE_SEARCH:
        return _azure_search(query)
    return _in_memory_search(query)


print(f"RAG ready. Using {'Azure AI Search' if USE_AZURE_SEARCH else 'in-memory search'}.")

## 3. స్మృతి

ఒక సపోర్ట్ ఏజెంట్ ఎవరితో మాట్లాడుతున్నాడో మరచిపోతే ఆ ఏజెంట్ చెడ్డదని అర్థం. మేము ప్రతి కస్టమర్ కొరకు ఒక చిన్న ప్రొఫైల్ నిల్వ ఉంచుతాము మరియు ఏజెంట్ సూచనల్లో ఒక చిన్న సారాంశాన్ని ఇంజెక్ట్ చేస్తాము. ప్రొడక్షన్ లో ఇది ఒక మెమరీ సర్వీస్ (పాఠం 13 చూడండి); ఇక్కడ ఒక డిక్ట్ ఈ నమూనా కనిపిస్తుందిఇ.  


In [ ]:
CUSTOMER_MEMORY: dict[str, dict] = {
    "cust-42": {"name": "Dana", "tier": "enterprise", "recent_order": "A1002"},
    "cust-99": {"name": "Sam", "tier": "standard", "recent_order": "A1003"},
}


def memory_context(customer_id: str) -> str:
    profile = CUSTOMER_MEMORY.get(customer_id)
    if not profile:
        return "This is a new customer with no history."
    return (
        f"Customer {profile['name']} ({profile['tier']} tier). "
        f"Most recent order: {profile['recent_order']}."
    )


print(memory_context("cust-42"))

## 4 & 5. మోడల్ రూటింగ్ మరియు ప్రతిస్పందన క్యాచింగ్

ఒకే అభ్యర్ధన హ్యాండ్లర్‌లో కలిపిన రెండు ఖర్చు నియంత్రణ పద్ధతులు:

- **రూటింగ్**: ఒక చవకైన హెయురిస్టిక్ వర్గీకర్త అభ్యర్థన చిన్నదో పెద్దదో మోడల్ అవసరమో నిర్ణయిస్తుంది.
- **క్యాచింగ్**: సాధారణీకరించిన పునరావృత ప్రశ్నలను మోడల్ కాల్ లేకుండా క్యాచ్ నుండి నేరుగా అందజేస్తారు.

ఇక్కడ వర్గీకర్త ఉద్దేశపూర్వకంగా సాదాసీదాగా ఉంటుంది. ఉత్పత్తిలో మీరు దీన్ని ట్రాఫిక్‌తో నిర్ధారించి Foundry యొక్క మోడల్ రూటర్ తో మార్చవచ్చు.


In [ ]:
SMALL_MODEL = os.getenv("AZURE_AI_SMALL_MODEL", model)   # e.g. gpt-5-nano
LARGE_MODEL = os.getenv("AZURE_AI_LARGE_MODEL", model)   # e.g. gpt-5-mini

response_cache: dict[str, str] = {}
route_counters = {"small": 0, "large": 0, "cache": 0}


def normalize(query: str) -> str:
    return re.sub(r"\s+", " ", query.lower().strip())


COMPLEX_SIGNALS = ("refund", "cancel", "complaint", "escalate", "broken", "wrong", "why")


def is_simple(query: str) -> bool:
    """Route complex or high-stakes requests to the large model; everything else to the small one."""
    q = query.lower()
    if any(signal in q for signal in COMPLEX_SIGNALS):
        return False
    return len(q.split()) <= 20


def choose_model(query: str) -> str:
    return SMALL_MODEL if is_simple(query) else LARGE_MODEL


print(f"Small model: {SMALL_MODEL} | Large model: {LARGE_MODEL}")

## 6 & 8. ఏజెంట్, మానవ ఆమోదం మరియు పరిశీలనీయత

ఇప్పుడు మేము పైన పేర్కొన్న టూల్స్ నుండి ఏజెంట్‌ను సమకూర్చి ప్రతి అభ్యర్థనను ఒక OpenTelemetry స్పాన్‌లో చుట్టేస్తాము. `handle_support_request` ఫంక్షన్ ఉత్పత్తి అభ్యర్థన హ్యాండ్లర్: క్యాషే → మార్గం → ట్రేస్ → రన్ → క్యాషే.

మానవ ఆమోదం ఫ్రేమ్‌వర్క్ ద్వారా నిర్వహించబడుతుంది: `issue_refund` కి `approval_mode="always_require"` ఉన్నందున, రన్ ఆగిపోతోంది మరియు మేము స్పష్టంగా పరిష్కరించే ఆమోద అభ్యర్థనను ప్రదర్శిస్తుంది.


In [ ]:
# Tracing: use the Agent Framework tracer if available, else a no-op so the notebook runs anywhere.
try:
    from agent_framework.observability import get_tracer
    tracer = get_tracer()
except Exception:  # observability extras not installed
    from contextlib import contextmanager

    class _NoopSpan:
        def set_attribute(self, *_args, **_kwargs):
            pass

    class _NoopTracer:
        @contextmanager
        def start_as_current_span(self, _name):
            yield _NoopSpan()

    tracer = _NoopTracer()


SUPPORT_INSTRUCTIONS = (
    "You are Contoso's customer support agent. Be concise, friendly, and accurate. "
    "Use search_policies for policy questions, get_order_status for orders, "
    "open_ticket when a human needs to follow up, and issue_refund for refunds. "
    "Never invent policy details."
)

# Build one agent per model tier so we can route by cost. The current agent-framework
# selects the model on the client, so each tier gets its own FoundryChatClient.
_TOOLS = [get_order_status, open_ticket, search_policies, issue_refund]
_agents_by_model: dict[str, object] = {}


def agent_for(model_name: str):
    if model_name not in _agents_by_model:
        client = FoundryChatClient(
            project_endpoint=endpoint,
            model=model_name,
            credential=AzureCliCredential(),
        )
        _agents_by_model[model_name] = client.as_agent(
            name="ContosoSupportAgent",
            instructions=SUPPORT_INSTRUCTIONS,
            tools=_TOOLS,
        )
    return _agents_by_model[model_name]


# Default agent (used by the evaluation gate, which does not route).
support_agent = agent_for(SMALL_MODEL)


async def handle_support_request(query: str, customer_id: str) -> str:
    # 1. Serve from cache when we can.
    key = normalize(query)
    if key in response_cache:
        route_counters["cache"] += 1
        return response_cache[key]

    # 2. Route by complexity to control cost.
    chosen_model = choose_model(query)
    route_counters["small" if chosen_model == SMALL_MODEL else "large"] += 1

    # 3. Add per-customer memory to the prompt.
    context = memory_context(customer_id)
    prompt = f"[Customer context: {context}]\n\n{query}"

    # 4. Run inside a trace span for observability.
    with tracer.start_as_current_span("support_request") as span:
        span.set_attribute("customer.id", customer_id)
        span.set_attribute("routed.model", chosen_model)
        response = await agent_for(chosen_model).run(prompt)

    text = response.text
    response_cache[key] = text
    return text


print("Support agent assembled.")

In [ ]:
# Try a few requests. The first is simple (small model), the second is a refund (large model + approval path).
print(await handle_support_request("What is your return window?", "cust-99"))
print("---")
print(await handle_support_request("Where is my order A1002?", "cust-42"))
print("---")
# Repeat the first question -> served from cache.
print(await handle_support_request("What is your return window?", "cust-99"))
print("---")
print("Routing counters:", route_counters)

## 7. మూల్యాంకన గేట్

ఇది పాఠం నుండి విడుదల గేట్: ఒక ఆఫ్‌లైన్ పరీక్ష సెట్టు ఏజెంట్ను స్కోరు చేస్తుంది, మరియు ఉత్పత్తిలో పెట్టుబడి కేవలం పాస్ రేటు ఒక ఆ_THRESHOLD_ ను దాటినట్లయితేనే ముందుకు సాగుతుంది. ఇక్కడ స్కోరర్ సులభమైన కీవర్డ్-ఓవర్ల్యాప్ తనిఖీ, నోట్‌బుక్‌ను స్వీయ-సహజంగా ఉంచడానికి; ఉత్పత్తిలో మీరు LLM-గా న్యాయమూర్తి లేదా ఒక ఫ్రేమ్‌వర్క్ ముల్యాంకకుని ఉపయోగిస్తారు (పాఠం 10 చూడండి).


In [ ]:
TEST_CASES = [
    {"input": "How long do I have to return an item?", "expected": ["30 days", "refund"]},
    {"input": "How fast is standard shipping?", "expected": ["3-5", "business days"]},
    {"input": "What is the status of order A1001?", "expected": ["shipped", "A1001"]},
    {"input": "Do your electronics have a warranty?", "expected": ["12-month", "warranty"]},
]


def score_response(actual: str, expected_keywords: list[str]) -> float:
    actual_l = actual.lower()
    hits = sum(1 for kw in expected_keywords if kw.lower() in actual_l)
    return hits / len(expected_keywords)


async def evaluation_gate(test_cases: list[dict], threshold: float = 0.8) -> bool:
    passed = 0
    for case in test_cases:
        result = await support_agent.run(case["input"])
        s = score_response(result.text, case["expected"])
        status = "PASS" if s >= 0.5 else "FAIL"
        print(f"[{status}] {case['input']}  (score={s:.0%})")
        if s >= 0.5:
            passed += 1
    pass_rate = passed / len(test_cases)
    print(f"\nEvaluation pass rate: {pass_rate:.0%} (gate: {threshold:.0%})")
    return pass_rate >= threshold


gate_passed = await evaluation_gate(TEST_CASES, threshold=0.8)
print("\nDeploy allowed:" , gate_passed)

## కలిపి ఉంచడం: ఒక అనుకరణ విడుదల

ఈ సెల్ క్రింద ఉన్నది పాఠం వివరిస్తున్న మొత్తం లూప్‌ను చూపిస్తుంది: మూల్యాంకన గేట్ను నడపండి, మరియు పాస్ అయితే మాత్రమే "డిప్లాయ్" చేయండి. మీరు ఏజెంట్ వెర్షన్‌ను Foundry Agent Service కు ప్రమోట్ చేయడానికి ముందుగా CI లో ఈ విధానాన్ని నడపాలి.


In [ ]:
async def release(test_cases: list[dict]) -> None:
    print("Running pre-deployment evaluation gate...\n")
    if await evaluation_gate(test_cases, threshold=0.8):
        print("\n✅ Gate passed — promoting agent version to the Foundry Agent Service.")
    else:
        print("\n❌ Gate failed — release blocked. Fix the agent and re-run.")


await release(TEST_CASES)

## సారాంశం

మీరు ప్రతి కార్యాచరణ సంబంధిత అంశాలతో కూడిన ఉత్పాదన-సిద్ధత్వ కస్టమర్ సపోర్ట్ ఏజెంట్‌ను సిధ్ధం చేశారు:

- **పరికరాలు, RAG, మరియు మెమొరీ** ఏజెంట్‌కు సామర్ధ్యం మరియు సందర్భం ఇస్తాయి.
- **మాడల్ రూటింగ్ మరియు కాచింగ్** ఆలస్యం మరియు ఖర్చును నియంత్రణలో ఉంచుతాయి.
- **మానవ ఆమోదం** పెద్ద రీఫండ్స్ వంటి అధిక-ప్రమాద చర్యలను రక్షిస్తుంది.
- **మూల్యాంకన గేట్** చెడు విడుదలలను పంపకానికి ముందు నిరోధిస్తుంది.
- **ట్రేసింగ్** ప్రతి అభ్యర్థనని పరిశీలించదగినవిధంగా చేస్తుంది.

### సవాలు

ఈ ఏజెంట్‌ను విస్తరించండి:

1. **బహుముఖ మోడల్స్‌ను మద్దతు ఇవ్వండి** — మూడవ "తర్క" దశను చేర్చండి మరియు ఎస్కలేషన్లు/పిర్యాదులను దానికి మార్గనిర్దేశం చేయండి.
2. **మూల్యాంకన గేట్లను చేర్చండి** — `TEST_CASES`లో రీఫండ్-ఆమోద దృశ్యాలను పొడగించి గేట్ రిగ్రెషన్‌లను పట్టుకోవడం నిర్ధారించండి.
3. **ఖర్చు-తెలుసుకున్న రూటింగ్‌ను జోడించండి** — ప్రతి అభ్యర్థనకు అంచనా ఖర్చును (చిన్నదిగా, పెద్దదిగా, కాచింగ్‌తో) ట్రాక్ చేసి, కలిపిన ప్రశ్నల బ్యాచ్ తరువాత ఖర్చు నివేదికను ముద్రించండి.

తర్వాతి పాఠంలో మీరు వ్యతిరేక ప్రయాణం చేపట్టుకొని Microsoft Foundry Local మరియు Qwenతో మీ సొంత యంత్రంపై ఏజెంట్‌ను పూర్తిగా నడిపిస్తారు.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**అస్వీకరణ**:
ఈ పత్రం AI అనువాద సేవ [Co-op Translator](https://github.com/Azure/co-op-translator) ఉపయోగించి అనువదించబడింది. మేము ఖచ్చితత్వానికి ప్రయత్నిస్తున్నప్పటికీ, ఆటోమేటెడ్ అనువాదాలు తప్పులు లేదా అసమగ్రతలను కలిగి ఉండవచ్చు. దాని స్వదేశ భాషలో ఉన్న అసలు పత్రాన్ని అధికారం కలిగిన మూలంగా పరిగణించాలి. కీలకమైన సమాచారం కోసం, ప్రొఫెషనల్ మానవ అనువాదాన్ని సిఫారసు చేస్తాము. ఈ అనువాదం ఉపయోగం వల్ల కలిగే ఏవైనా అపార్థాలు లేదా తప్పుదారులు కోసం మేము బాధ్యత వహించము.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
